# 03b - Merge COPD Cohort with RAAS Exposures

Short summary:
[03b - Merge COPD Cohort with RAAS Exposures (SHORT)](../docs/03b_merge_exposures_SHORT.md)

---

## 1. Objective
Merge the COPD ICU cohort with ACE inhibitor, ARB, and any-class and both-class prescription-order indicators and validate the resulting exposure-enhanced cohort.

---

## 2. Data Processing Steps

<!-- Generated notebook input context -->

## Inputs and execution mode

Input definitions: [BigQuery mimic-iv-portfolio.copd_raas.cohort_copd / cohort_copd_with_raas; 03_merge_exposures.sql](../sql/03_merge_exposures.sql).

New execution uses the queries and shared loader shown in the code. The stored outputs below were refreshed from saved clinical aggregates, without a new clinical query or model refit.

Stored-output evidence: [original clinical execution](../docs/NOTEBOOK_EXECUTION_VALIDATION.json) · [presentation refresh](../docs/NOTEBOOK_PRESENTATION_VALIDATION.json).

### 2.1 Connect to BigQuery


In [2]:
# Shared ADC client with US location, dry runs and per-job byte guards.
from pathlib import Path
import sys
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from notebook_bq_runtime import client_for_notebook, run_sql_script, query_to_df
client = client_for_notebook()
print("Bounded BigQuery helpers configured for mimic-iv-portfolio (US); see the execution or presentation record for input origin.")
import notebook_presentation as presentation


Bounded BigQuery helpers configured for mimic-iv-portfolio (US); see the execution or presentation record for input origin.


### 2.2 Execute Merge SQL


In [3]:
# Run merge SQL to create cohort_copd_with_raas in BigQuery
run_sql_script("../sql/03_merge_exposures.sql")

Presentation replay of saved construction evidence: 03_merge_exposures.sql (SQL not executed)


### 2.3 Row Count Validation


In [4]:
# Sanity check: row counts before and after merge

sql_row_counts = """
SELECT
  'cohort_copd' AS table_name,
  COUNT(*) AS n
FROM `mimic-iv-portfolio.copd_raas.cohort_copd`
UNION ALL
SELECT
  'cohort_copd_with_raas' AS table_name,
  COUNT(*) AS n
FROM `mimic-iv-portfolio.copd_raas.cohort_copd_with_raas`
"""
row_counts = query_to_df(sql_row_counts)
presentation.display_table("03b", "merge", row_counts, presentation.merge_summary(row_counts))

table_name,n
cohort_copd,11964
cohort_copd_with_raas,11964


The row-count comparison and its interpretation are generated from the same table above.

### 2.4 Missing Exposure Checks


In [5]:
# Sanity check: count NULLs in key exposure variables
sql_nulls = """
SELECT
  SUM(CASE WHEN acei_pre_icu IS NULL THEN 1 ELSE 0 END) AS null_acei_pre_icu,
  SUM(CASE WHEN arb_pre_icu IS NULL THEN 1 ELSE 0 END) AS null_arb_pre_icu,
  SUM(CASE WHEN raas_any_pre_icu IS NULL THEN 1 ELSE 0 END) AS null_any_pre_icu,
  SUM(CASE WHEN exposure_group_4cat IS NULL THEN 1 ELSE 0 END) AS null_group
FROM `mimic-iv-portfolio.copd_raas.cohort_copd_with_raas`
"""
missing_counts = query_to_df(sql_nulls)
presentation.display_table("03b", "missing", missing_counts, presentation.missing_summary(missing_counts))

null_acei_pre_icu,null_arb_pre_icu,null_any_pre_icu,null_group
0,0,0,0


The missing-field totals and interpretation above are calculated from the current aggregate result.

### 2.5 Exposure Distribution Checks


The distribution of the four-category RAAS exposure variable was examined.

In [6]:
# Sanity check: distribution of exposure groups
sql_grp = """
SELECT exposure_group_4cat, COUNT(*) AS n
FROM `mimic-iv-portfolio.copd_raas.cohort_copd_with_raas`
GROUP BY exposure_group_4cat
ORDER BY exposure_group_4cat
"""
group_counts = query_to_df(sql_grp)
presentation.display_table("03b", "groups", group_counts, presentation.group_summary(group_counts))

exposure_group_4cat,n
acei_only,643
arb_only,290
both,13
neither,11018


### 2.6 Cross-check of RAAS Inhibitor Exposure Counts


In [7]:
# Sanity check: counts of users in each RAAS inhibitor category
sql_cross = """
SELECT
  SUM(CASE WHEN acei_pre_icu = 1 THEN 1 ELSE 0 END) AS acei_users,
  SUM(CASE WHEN arb_pre_icu = 1 THEN 1 ELSE 0 END) AS arb_users,
  SUM(CASE WHEN raas_any_pre_icu = 1 THEN 1 ELSE 0 END) AS any_users
FROM `mimic-iv-portfolio.copd_raas.cohort_copd_with_raas`
"""

order_counts = query_to_df(sql_cross)
presentation.display_table("03b", "orders", order_counts, presentation.orders_summary(order_counts))

acei_users,arb_users,any_users
656,303,946


The class-flag totals and numerical summary above are generated from the same aggregate query.

---

## 3. Output Dataset

**Input Dataset**
- `mimic-iv-portfolio.copd_raas.cohort_copd`
- `mimic-iv-portfolio.copd_raas.cohort_copd_raas_detailed`

**Output Dataset**
- `mimic-iv-portfolio.copd_raas.cohort_copd_with_raas`

**Downstream Usage**
- This exposure-enhanced cohort table supports exposure distribution checks and documents the detailed ACE inhibitor, ARB, and any-class and both-class prescription-order merge.


---

## Next Step

After phase 03, run the complete `sql/04_build_outcomes.sql` script in BigQuery US and wait for its QA and output table `cohort_copd_outcomes`. Then run `sql/04_build_outcomes_extended.sql` to create `cohort_copd_outcomes_extended`.

The shared loader for 04a–04d reads the extended table and joins `cohort_copd_raas_detailed`. `cohort_copd_with_raas` supports exposure checks but does not replace either outcome table. The next notebook fits models; it does not construct these tables.

- [04a — Primary survival analysis](04a_outcomes_and_modeling.ipynb)
- [Outcome-table dependencies](../docs/METHODS_OVERVIEW.md#outcome-table-dependencies)
